# Phase 7 Regression Evaluation Walkthrough

This notebook demonstrates the Task 4 mocked 2-scenario by 2-repetition regression path using the in-process runner, mocked agent target, ingestion service, and evaluation service.

Prerequisites from the repository root:

```bash
cp .env.example .env
docker compose up -d --wait postgres
uv run alembic upgrade head
```


In [ ]:
from collections.abc import Iterable
from pathlib import Path
import sys

from sqlalchemy import delete, select
from sqlalchemy.ext.asyncio import async_sessionmaker

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from obs_platform.config import DatabaseOnlySettings
from obs_platform.database import create_engine, wait_for_database
from obs_platform.db.models import (
    AgentRun,
    EvaluationResult,
    JudgeCall,
    LLMCall,
    RegressionRun,
    RunFailure,
    Span,
    ToolCall,
)
from obs_platform.regressions.persistence import create_regression_run
from obs_platform.regressions.runner import MockedAgentTarget, RegressionRunner
from obs_platform.telemetry.v1 import load_fixture


SCENARIO_IDS = ["GS-DEBUG-SMOKE-01", "GS-DEBUG-TRAJ-01"]


async def delete_runs(session, run_ids: Iterable[str]) -> None:
    run_ids = list(run_ids)
    if not run_ids:
        return
    for model in (
        RunFailure,
        EvaluationResult,
        JudgeCall,
        LLMCall,
        ToolCall,
        Span,
        AgentRun,
    ):
        await session.execute(delete(model).where(model.run_id.in_(run_ids)))
    await session.commit()


async def run_mocked_regression(session, name: str) -> list[tuple[str, int, str]]:
    regression = await create_regression_run(
        session,
        name=name,
        agent_version="agent-v1",
        agent_model_provider="mock-provider",
        agent_model_name="mock-model",
        prompt_version="prompt-v1",
        repetitions=2,
        scenario_ids=SCENARIO_IDS,
    )
    runner = RegressionRunner(
        session=session,
        target=MockedAgentTarget(
            {
                "GS-DEBUG-SMOKE-01": load_fixture("healthy_success"),
                "GS-DEBUG-TRAJ-01": load_fixture("trajectory_error"),
            }
        ),
    )
    result = await runner.run(regression.id)
    rows = (
        await session.execute(
            select(
                AgentRun.scenario_id,
                AgentRun.repetition_index,
                RunFailure.overall_status,
            )
            .join(RunFailure, RunFailure.run_id == AgentRun.run_id)
            .where(AgentRun.regression_run_id == regression.id)
            .order_by(AgentRun.scenario_id, AgentRun.repetition_index)
        )
    ).all()
    await delete_runs(session, result.created_run_ids)
    await session.execute(
        delete(RegressionRun).where(RegressionRun.id == regression.id)
    )
    await session.commit()
    return [
        (scenario_id, repetition_index, status)
        for scenario_id, repetition_index, status in rows
    ]


engine = create_engine(DatabaseOnlySettings().db)
await wait_for_database(engine)
Session = async_sessionmaker(engine, expire_on_commit=False)

async with Session() as session:
    first = await run_mocked_regression(session, "notebook-phase-7-task-4-a")
    second = await run_mocked_regression(session, "notebook-phase-7-task-4-b")

await engine.dispose()

assert first == second
first
